In [1]:
import os
import json
import pandas as pd
import glob
from collections import defaultdict
import numpy as np


def get_results_data(results_dir: str, datasets: list, models: list):

    datasets = set(datasets)
    models = set(models)

    data = defaultdict(lambda: defaultdict(dict))

    for folder in os.listdir(results_dir):
        folder_path = os.path.join(results_dir, folder)

        setup_files = glob.glob(f"{folder_path}/setup*.json")
        if not setup_files:
            continue

        with open(setup_files[0], "r") as f:
            setup = json.load(f)

        dataset_name = setup["database_name"]
        model_name = setup["model_name"]
        k_fold_position = setup["split_position"]

        if dataset_name not in datasets or model_name not in models:
            continue

        history_file = glob.glob(f"{folder_path}/*history*.json")[0]

        with open(history_file, "r") as f:
            history = json.load(f)

        history = sorted(history, key=lambda x: x["epoch"])

        data[dataset_name][model_name][int(k_fold_position)] = history

    data = {
        d: {m: dict(folds) for m, folds in model_dict.items()}
        for d, model_dict in data.items()
    }

    return data

def convert_data_to_df(data):
    rows = []

    for dataset, models in data.items():
        for model, folds in models.items():
            for fold, epochs in folds.items():
                for e in epochs:
                    rows.append({
                        "dataset": dataset,
                        "model": model,
                        "fold": fold,
                        "epoch": e["epoch"],
                        "loss_fit": e["loss_fit"],
                        "loss_val": e["loss_val"]
                    })

    df = pd.DataFrame(rows)
    return df

def compute_loss_delta(group):
    group = group.sort_values("epoch")
    first_loss = group["loss_val"].iloc[0]
    last_loss = group["loss_val"].iloc[-5]
    return (first_loss - last_loss) / first_loss

In [2]:
runs_path = "../runs/proposal"

datasets = ["amazon-movies-tvs", "jester-joke", "ml-1m"]
models = ["mf", "dgat", "lightgcn", "prgat", "prlightgcn", "ordrec", "cbpmf", "cpmf", "lbd"]

In [3]:
df = convert_data_to_df(get_results_data(runs_path, datasets, models))

In [4]:
loss_deltas = (
    df.groupby(["dataset", "model", "fold"], group_keys=False)
    .apply(compute_loss_delta)
    .reset_index(name="loss_delta")
)

loss_delta_stats = (
    loss_deltas.groupby(["dataset", "model"])["loss_delta"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={
        "mean": "loss_delta_mean",
        "std": "loss_delta_std"
    })
)

/tmp/ipykernel_15073/2419971969.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(compute_loss_delta)


In [5]:
loss_delta_stats

,dataset,model,loss_delta_mean,loss_delta_std
0,amazon-movies-tvs,cbpmf,0.811326,0.158206
1,amazon-movies-tvs,cpmf,0.390047,0.199819
2,amazon-movies-tvs,dgat,0.011542,0.001517
3,amazon-movies-tvs,lbd,-0.006601,0.000458
4,amazon-movies-tvs,lightgcn,0.443604,0.004507
5,amazon-movies-tvs,mf,0.091428,0.002831
6,amazon-movies-tvs,ordrec,0.000000,0.000000
7,amazon-movies-tvs,prgat,0.014157,0.001704
8,amazon-movies-tvs,prlightgcn,0.213153,0.004517
9,jester-joke,cbpmf,0.082631,0.052307
